In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.neural_network import MLPClassifier
from collections import Counter

# ==============================================================================
# 1. Configuration & Setup
# ==============================================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAMES = {
    "mBERT": "bert-base-multilingual-cased",
    "IndicBERT": "ai4bharat/indic-bert",
    "XLMR": "xlm-roberta-base"
}

# Each scale = (tfidf_latent_dim, transformer_dim, total_fused_dim)
SCALES = {
    512:  {"tfidf_svd": 512,  "tfidf_latent": 32,  "transformer_dim": 480},
    1024: {"tfidf_svd": 1024, "tfidf_latent": 64,  "transformer_dim": 960},
    2048: {"tfidf_svd": 2048, "tfidf_latent": 128, "transformer_dim": 1920},
}

LABELS = ["Substantiated", "Sarcastic", "Opinionated", "Positive", "Negative", "Neutral", "None of the above"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label  = {i: label for label, i in label2id.items()}

# ==============================================================================
# 2. Load Data
# ==============================================================================
train_df   = pd.read_csv("PS_train.csv")
test_df    = pd.read_csv("PS_test_without_labels.csv")
tfidf_train = pd.read_csv("tfidf_train_features.csv").values   # high-dim sparse TF-IDF
tfidf_test  = pd.read_csv("tfidf_test_features.csv").values

y_train = train_df["labels"].map(label2id).values
y_test  = test_df["labels"].map(label2id).values

# ==============================================================================
# 3. TF-IDF Branch: SVD + MLP compression at each scale
#    Produces latent vectors of size 32, 64, 128 for scales 512, 1024, 2048
# ==============================================================================
print("=== TF-IDF Branch: SVD + MLP compression ===")

tfidf_latents = {}   # key: scale (512/1024/2048) -> (train_latent, test_latent)

for fused_dim, cfg in SCALES.items():
    svd_dim     = cfg["tfidf_svd"]
    latent_dim  = cfg["tfidf_latent"]

    # Step 1: SVD dimensionality reduction
    n_components = min(svd_dim, tfidf_train.shape[1], tfidf_train.shape[0])
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    tfidf_svd_train = svd.fit_transform(tfidf_train)
    tfidf_svd_test  = svd.transform(tfidf_test)

    scaler_svd = StandardScaler()
    tfidf_svd_train = scaler_svd.fit_transform(tfidf_svd_train)
    tfidf_svd_test  = scaler_svd.transform(tfidf_svd_test)

    # Step 2: MLP auto-encoder style compression to latent_dim
    mlp = MLPClassifier(
        hidden_layer_sizes=(latent_dim,),
        activation="relu",
        max_iter=300,
        random_state=42
    )
    mlp.fit(tfidf_svd_train, y_train)

    # Extract the first hidden layer activations as the latent embedding
    def get_mlp_latent(mlp, X):
        layer_input = X
        for i, (W, b) in enumerate(zip(mlp.coefs_, mlp.intercepts_)):
            layer_input = np.dot(layer_input, W) + b
            if i == 0:                          # stop after first hidden layer
                layer_input = np.maximum(layer_input, 0)  # ReLU
                break
        return layer_input

    tfidf_latents[fused_dim] = (
        get_mlp_latent(mlp, tfidf_svd_train),
        get_mlp_latent(mlp, tfidf_svd_test)
    )

# ==============================================================================
# 4. Transformer Branch: extract contextual embeddings, then PCA to target dims
#    Produces embeddings of 480, 960, 1920 for scales 512, 1024, 2048
# ==============================================================================

def extract_embeddings(model_name, texts):
    """Mean-pool transformer last hidden states (ignoring padding)."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModel.from_pretrained(model_name).to(DEVICE)
    model.eval()

    all_embs = []
    batch_size = 32
    for i in range(0, len(texts), batch_size):
        batch  = texts[i: i + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=128, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = model(**inputs)
            mask   = inputs["attention_mask"].unsqueeze(-1) \
                           .expand(outputs.last_hidden_state.size()).float()
            pooled = (torch.sum(outputs.last_hidden_state * mask, 1) /
                      torch.clamp(mask.sum(1), min=1e-9))
            all_embs.append(pooled.cpu().numpy())
    return np.vstack(all_embs)

# ==============================================================================
# 5. Main Loop: fuse features at each scale, train classifier, collect predictions
# ==============================================================================
# Structure: all_preds[scale][model_nickname] = array of predicted label ids
all_preds = {scale: {} for scale in SCALES}

for nickname, model_path in MODEL_NAMES.items():
    base_train = extract_embeddings(model_path, train_df["content"].tolist())
    base_test  = extract_embeddings(model_path, test_df["content"].tolist())

    for fused_dim, cfg in SCALES.items():
        trans_dim  = cfg["transformer_dim"]
        latent_dim = cfg["tfidf_latent"]


        # --- Transformer: PCA to target transformer_dim ---
        from sklearn.decomposition import PCA
        n_components = min(trans_dim, base_train.shape[1], base_train.shape[0])
        pca = PCA(n_components=n_components, random_state=42)
        trans_train = pca.fit_transform(base_train)
        trans_test  = pca.transform(base_test)

        scaler_trans = StandardScaler()
        trans_train  = scaler_trans.fit_transform(trans_train)
        trans_test   = scaler_trans.transform(trans_test)

        # --- Concatenate: TF-IDF latent ⊕ Transformer embeddings ---
        tfidf_lat_train, tfidf_lat_test = tfidf_latents[fused_dim]
        X_train = np.hstack([tfidf_lat_train, trans_train])   # (N, fused_dim)
        X_test  = np.hstack([tfidf_lat_test,  trans_test])

        # --- Train SVC classifier ---
        clf = SVC(kernel="linear", random_state=42)
        clf.fit(X_train, y_train)
        all_preds[fused_dim][nickname] = clf.predict(X_test)

# ==============================================================================
# 6. Two-Tier Majority Voting Ensemble
# ==============================================================================

# --- Tier 1: Intra-Scale Voting ---
# For each scale, majority vote across mBERT, IndicBERT, XLM-RoBERTa
scale_consensus = {}   # scale -> consensus prediction array

for fused_dim, model_preds in all_preds.items():
    preds_matrix = np.array(list(model_preds.values()))  # (3, N)
    consensus = []
    for i in range(preds_matrix.shape[1]):
        vote = Counter(preds_matrix[:, i]).most_common(1)[0][0]
        consensus.append(vote)
    scale_consensus[fused_dim] = np.array(consensus)

    # print(classification_report(y_test, consensus, target_names=LABELS))

# --- Tier 2: Inter-Scale Voting ---
# Majority vote across the three scale-level consensus predictions
consensus_matrix = np.array(list(scale_consensus.values()))  # (3, N)
final_preds = []
for i in range(consensus_matrix.shape[1]):
    final_vote = Counter(consensus_matrix[:, i]).most_common(1)[0][0]
    final_preds.append(final_vote)

# ==============================================================================
# 7. Save Results
# ==============================================================================
test_df["final_prediction"] = [id2label[p] for p in final_preds]
test_df.to_csv("Wise_political_run_1.csv", index=False)
print("\nWise_political_run_1.csv")


Wise_political_run_1.csv
